<a href="https://colab.research.google.com/github/Ayseatci/DI725_Project/blob/dev/notebooks/03_phase2/SWIN_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Image Model: SWIN V2

In [1]:
!pip install -q timm transformers accelerate

In [3]:
!pip install -q timm transformers accelerate wandb open_clip_torch huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.8 MB/s eta 0:00:00


In [4]:
import os
import re
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

from torchvision import transforms
import matplotlib.pyplot as plt

import wandb
import open_clip
from huggingface_hub import hf_hub_download

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
!unzip -q /content/drive/MyDrive/DI725_project_dataset_nomask.zip

In [7]:
LOCAL_DATA_ROOT = "/content/DI725_project_dataset_nomask"
LOCAL_CSV_PATH = os.path.join(LOCAL_DATA_ROOT, "captions.csv")
LOCAL_IMAGE_DIR = os.path.join(LOCAL_DATA_ROOT, "images")

os.makedirs(LOCAL_DATA_ROOT, exist_ok=True)

CSV_PATH = LOCAL_CSV_PATH
IMAGE_DIR = LOCAL_IMAGE_DIR

print("CSV:", CSV_PATH)
print("Images:", IMAGE_DIR)
print("Number of images:", len(os.listdir(IMAGE_DIR)))

CSV: /content/DI725_project_dataset_nomask/captions.csv
Images: /content/DI725_project_dataset_nomask/images
Number of images: 10000


In [8]:
CONFIG = {
    "sample_size": 10000,
    "img_size": 256,

    "batch_size": 32,
    "epochs": 15,
    "lr": 1e-4,
    "weight_decay": 1e-4,

    "model_name": "swinv2_tiny_window8_256",

    "early_stopping_patience": 3,
    "grad_clip": 1.0,

    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 42,

    "text_scale": 0.7
}

TARGET_COLS = ["Tree", "Shrub", "Grass", "Crop", "Built-up", "Barren", "Water"]

In [9]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CONFIG["seed"])

Data Loading and Text Cleaning

In [10]:
df = pd.read_csv(CSV_PATH)

TEXT_COLUMNS_RAW = [
    "hybrid_gemma3-4b",
    "text_qwen3-4b",
    "vision_gemma3-4b"
]

def clean_caption_no_numbers(text):
    text = str(text).lower()

    # remove percentages: 53%, 53 percent, 53 percentage
    text = re.sub(r"\b\d+(\.\d+)?\s*%|\b\d+(\.\d+)?\s*percent(age)?", "", text)

    # remove standalone numbers
    text = re.sub(r"\b\d+(\.\d+)?\b", "", text)

    leakage_words = [
        "covering", "coverage", "accounts for", "occupies",
        "making up", "comprises", "constitutes", "represents",
        "estimated", "approximately", "around", "about"
    ]

    for word in leakage_words:
        text = text.replace(word, "")

    text = re.sub(r"[%(),:;]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


for col in TEXT_COLUMNS_RAW:
    clean_col = col.replace("-", "_") + "_clean"
    df[clean_col] = df[col].apply(clean_caption_no_numbers)

    print(clean_col)
    print(
        "Remaining numeric leakage:",
        df[clean_col].str.contains(
            r"\d+\s*%|\d+\s*percent|\b\d+(\.\d+)?\b",
            regex=True,
            case=False
        ).sum()
    )

hybrid_gemma3_4b_clean
Remaining numeric leakage: 0


/tmp/ipykernel_6012/2462767828.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[clean_col].str.contains(


text_qwen3_4b_clean
Remaining numeric leakage: 0


/tmp/ipykernel_6012/2462767828.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[clean_col].str.contains(


vision_gemma3_4b_clean
Remaining numeric leakage: 0


/tmp/ipykernel_6012/2462767828.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[clean_col].str.contains(


In [11]:
df["hybrid_gemma3-4b"].head(10)

,hybrid_gemma3-4b
0,The image depicts a landscape dominated by ext...
1,The image depicts a largely arid landscape dom...
2,The image depicts a landscape dominated by ext...
3,The image depicts a valley dominated by dense ...
4,The image depicts a coastal area dominated by ...
5,The image depicts a landscape dominated by cul...
6,The image depicts a landscape dominated by a r...
7,The image depicts a hilly landscape dominated ...
8,The image depicts a landscape dominated by ext...
9,The image depicts a landscape dominated by ext...


Stratified Sampling and Split

In [25]:
needed_cols = (
    ["filename"]
    + TARGET_COLS
    + ["hybrid_gemma3_4b_clean", "text_qwen3_4b_clean", "vision_gemma3_4b_clean"]
)

df = df.dropna(subset=needed_cols).copy()

df["dominant_class"] = df[TARGET_COLS].idxmax(axis=1)

sample_df, _ = train_test_split(
    df,
    train_size= 0.9,
    stratify=df["dominant_class"],
    random_state=CONFIG["seed"]
)

sample_df = sample_df.reset_index(drop=True)

train_df, temp_df = train_test_split(
    sample_df,
    test_size=0.20,
    stratify=sample_df["dominant_class"],
    random_state=CONFIG["seed"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["dominant_class"],
    random_state=CONFIG["seed"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 7200
Val: 900
Test: 900


In [26]:
def check_target_distribution(train_df, val_df, test_df, target_cols):
    summary = []

    for name, data in [
        ("train", train_df),
        ("val", val_df),
        ("test", test_df)
    ]:
        row = {"split": name, "n": len(data)}

        for col in target_cols:
            row[f"{col}_mean"] = data[col].mean()
            row[f"{col}_std"] = data[col].std()

        summary.append(row)

    return pd.DataFrame(summary)

dist_summary = check_target_distribution(train_df, val_df, test_df, TARGET_COLS)
dist_summary

,split,n,Tree_mean,Tree_std,Shrub_mean,Shrub_std,Grass_mean,Grass_std,Crop_mean,Crop_std,Built-up_mean,Built-up_std,Barren_mean,Barren_std,Water_mean,Water_std
0,train,7200,28.719861,35.213752,0.840694,4.123632,45.128611,34.031417,18.201806,30.781482,1.121250,5.276612,4.280833,11.677308,1.706944,9.437475
1,val,900,28.857778,35.041653,0.695556,3.620880,46.078889,33.780114,17.412222,29.600192,1.245556,5.615877,4.198889,11.507670,1.511111,8.727164
2,test,900,28.611111,34.286367,0.934444,4.449086,46.344444,34.061536,17.255556,30.359823,1.062222,5.231941,4.067778,10.844172,1.724444,9.413245


In [27]:
dominant_dist = pd.DataFrame({
    "train": train_df["dominant_class"].value_counts(normalize=True),
    "val": val_df["dominant_class"].value_counts(normalize=True),
    "test": test_df["dominant_class"].value_counts(normalize=True)
}).fillna(0)

dominant_dist

,train,val,test
dominant_class,,,
Grass,0.470417,0.470000,0.470000
Tree,0.303611,0.304444,0.303333
Crop,0.180417,0.180000,0.180000
Barren,0.019306,0.018889,0.020000
Water,0.017778,0.017778,0.017778
Built-up,0.005833,0.006667,0.005556
Shrub,0.002639,0.002222,0.003333


Transforms

In [ ]:
IMG_SIZE = CONFIG["img_size"]

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

Dataset Classes

In [ ]:
class LandCoverDataset(Dataset):
    def __init__(self, df, image_dir, transform=None, tokenizer=None, text_col=None, max_len=128):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.tokenizer = tokenizer
        self.text_col = text_col
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        target = torch.tensor(
            row[TARGET_COLS].values.astype(np.float32),
            dtype=torch.float32
        )

        item = {
            "image": image,
            "target": target
        }

        if self.tokenizer is not None and self.text_col is not None:
            text = str(row[self.text_col])

            enc = self.tokenizer(
                text,
                padding="max_length",
                truncation=True,
                max_length=self.max_len,
                return_tensors="pt"
            )

            item["input_ids"] = enc["input_ids"].squeeze(0)
            item["attention_mask"] = enc["attention_mask"].squeeze(0)

        return item

In [ ]:
class LandCoverRawTextDataset(Dataset):
    def __init__(self, df, image_dir, transform=None, text_col=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.text_col = text_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        text = str(row[self.text_col])

        target = torch.tensor(
            row[TARGET_COLS].values.astype(np.float32),
            dtype=torch.float32
        )

        return {
            "image": image,
            "text": text,
            "target": target
        }

Swin image-only

In [ ]:
class ImageOnlyModel(nn.Module):
    def __init__(self, model_name, num_outputs=7):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0
        )

        image_dim = self.backbone.num_features

        self.regressor = nn.Sequential(
            nn.LayerNorm(image_dim),
            nn.Linear(image_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, num_outputs)
        )

    def forward(self, image):
        image_features = self.backbone(image)
        return self.regressor(image_features)

Swin + MiniLM

In [ ]:
class ImageTextGatedFrozenScaledModel(nn.Module):
    def __init__(
        self,
        image_model_name,
        text_model_name,
        num_outputs=7,
        text_scale=1.0
    ):
        super().__init__()

        self.text_scale = text_scale

        self.image_backbone = timm.create_model(
            image_model_name,
            pretrained=True,
            num_classes=0
        )

        image_dim = self.image_backbone.num_features

        self.text_backbone = AutoModel.from_pretrained(text_model_name)

        for p in self.text_backbone.parameters():
            p.requires_grad = False

        text_dim = self.text_backbone.config.hidden_size

        self.text_proj = nn.Linear(text_dim, image_dim)

        self.gate_layer = nn.Sequential(
            nn.Linear(image_dim + image_dim, image_dim),
            nn.Sigmoid()
        )

        self.regressor = nn.Sequential(
            nn.LayerNorm(image_dim),
            nn.Linear(image_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, num_outputs)
        )

    def mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output.last_hidden_state
        mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

        return torch.sum(token_embeddings * mask, dim=1) / torch.clamp(
            mask.sum(dim=1),
            min=1e-9
        )

    def forward(self, image, input_ids, attention_mask):
        image_features = self.image_backbone(image)

        with torch.no_grad():
            text_output = self.text_backbone(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        text_features = self.mean_pooling(text_output, attention_mask)
        text_features = self.text_proj(text_features)

        gate_input = torch.cat([image_features, text_features], dim=1)
        gate = self.gate_layer(gate_input)

        fused_features = image_features + self.text_scale * gate * text_features

        return self.regressor(fused_features)

Remote CLIP Text Encoder

In [ ]:
class RemoteCLIPTextEncoder(nn.Module):
    def __init__(self, model_name="ViT-B-32"):
        super().__init__()

        self.model, _, _ = open_clip.create_model_and_transforms(
            model_name,
            pretrained=None
        )

        self.tokenizer = open_clip.get_tokenizer(model_name)

        ckpt_path = hf_hub_download(
            repo_id="chendelong/RemoteCLIP",
            filename="RemoteCLIP-ViT-B-32.pt"
        )

        ckpt = torch.load(ckpt_path, map_location="cpu")

        if "state_dict" in ckpt:
            ckpt = ckpt["state_dict"]

        cleaned = {}
        for k, v in ckpt.items():
            k = k.replace("module.", "").replace("model.", "")
            cleaned[k] = v

        self.model.load_state_dict(cleaned, strict=False)

        print("RemoteCLIP weights loaded")

        for p in self.model.parameters():
            p.requires_grad = False

        self.model.eval()

    def forward(self, texts):
        device = next(self.model.parameters()).device
        tokens = self.tokenizer(list(texts)).to(device)

        with torch.no_grad():
            features = self.model.encode_text(tokens)
            features = features / features.norm(dim=-1, keepdim=True)

        return features

In [ ]:
class SwinV2RemoteCLIPFusion(nn.Module):
    def __init__(
        self,
        image_model_name,
        num_outputs=7,
        text_scale=0.7
    ):
        super().__init__()

        self.text_scale = text_scale

        self.image_backbone = timm.create_model(
            image_model_name,
            pretrained=True,
            num_classes=0
        )

        image_dim = self.image_backbone.num_features

        self.text_encoder = RemoteCLIPTextEncoder(model_name="ViT-B-32")

        with torch.no_grad():
            dummy_text = ["satellite image of land cover"]
            dummy_features = self.text_encoder(dummy_text)
            text_dim = dummy_features.shape[1]

        self.text_proj = nn.Linear(text_dim, image_dim)

        self.gate_layer = nn.Sequential(
            nn.Linear(image_dim + image_dim, image_dim),
            nn.Sigmoid()
        )

        self.regressor = nn.Sequential(
            nn.LayerNorm(image_dim),
            nn.Linear(image_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, num_outputs)
        )

    def forward(self, image, texts):
        image_features = self.image_backbone(image)

        text_features = self.text_encoder(texts)
        text_features = text_features.to(image_features.device)
        text_features = self.text_proj(text_features)

        gate_input = torch.cat([image_features, text_features], dim=1)
        gate = self.gate_layer(gate_input)

        fused = image_features + self.text_scale * gate * text_features

        return self.regressor(fused)

Evaluation functions

In [ ]:
@torch.no_grad()
def evaluate_image_model(model, loader, config):
    model.eval()

    criterion = nn.L1Loss()
    total_loss = 0
    all_preds = []
    all_targets = []

    for batch in loader:
        images = batch["image"].to(config["device"])
        targets = batch["target"].to(config["device"])

        preds = model(images)
        loss = criterion(preds, targets)

        total_loss += loss.item() * images.size(0)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    class_mae = np.mean(np.abs(y_true - y_pred), axis=0)

    return total_loss / len(loader.dataset), mae, rmse, class_mae

In [ ]:
@torch.no_grad()
def evaluate_text_model(model, loader, config):
    model.eval()

    criterion = nn.L1Loss()
    total_loss = 0
    all_preds = []
    all_targets = []

    for batch in loader:
        images = batch["image"].to(config["device"])
        input_ids = batch["input_ids"].to(config["device"])
        attention_mask = batch["attention_mask"].to(config["device"])
        targets = batch["target"].to(config["device"])

        preds = model(images, input_ids, attention_mask)
        loss = criterion(preds, targets)

        total_loss += loss.item() * images.size(0)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    class_mae = np.mean(np.abs(y_true - y_pred), axis=0)

    return total_loss / len(loader.dataset), mae, rmse, class_mae

In [ ]:
@torch.no_grad()
def evaluate_remoteclip_model(model, loader, config):
    model.eval()

    criterion = nn.L1Loss()
    total_loss = 0
    all_preds = []
    all_targets = []

    for batch in loader:
        images = batch["image"].to(config["device"])
        texts = batch["text"]
        targets = batch["target"].to(config["device"])

        preds = model(images, texts)
        loss = criterion(preds, targets)

        total_loss += loss.item() * images.size(0)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    class_mae = np.mean(np.abs(y_true - y_pred), axis=0)

    return total_loss / len(loader.dataset), mae, rmse, class_mae

W&B Training Functions

In [ ]:
def train_image_model(model, train_loader, val_loader, config):
    criterion = nn.L1Loss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )

    best_val_mae = float("inf")
    best_state = None
    patience_counter = 0
    history = []

    for epoch in range(config["epochs"]):
        model.train()
        train_loss = 0

        for batch in train_loader:
            optimizer.zero_grad()

            images = batch["image"].to(config["device"])
            targets = batch["target"].to(config["device"])

            preds = model(images)
            loss = criterion(preds, targets)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
            optimizer.step()

            train_loss += loss.item() * images.size(0)

        train_loss /= len(train_loader.dataset)

        val_loss, val_mae, val_rmse, _ = evaluate_image_model(model, val_loader, config)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse
        })

        print(
            f"Epoch {epoch+1}/{config['epochs']} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val MAE: {val_mae:.4f} | "
            f"Val RMSE: {val_rmse:.4f}"
        )

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse
        })

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= config["early_stopping_patience"]:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_state)

    return model, pd.DataFrame(history)

In [ ]:
def train_text_model(model, train_loader, val_loader, config):
    criterion = nn.L1Loss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )

    best_val_mae = float("inf")
    best_state = None
    patience_counter = 0
    history = []

    for epoch in range(config["epochs"]):
        model.train()
        train_loss = 0

        for batch in train_loader:
            optimizer.zero_grad()

            images = batch["image"].to(config["device"])
            input_ids = batch["input_ids"].to(config["device"])
            attention_mask = batch["attention_mask"].to(config["device"])
            targets = batch["target"].to(config["device"])

            preds = model(images, input_ids, attention_mask)
            loss = criterion(preds, targets)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
            optimizer.step()

            train_loss += loss.item() * images.size(0)

        train_loss /= len(train_loader.dataset)

        val_loss, val_mae, val_rmse, _ = evaluate_text_model(model, val_loader, config)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse
        })

        print(
            f"Epoch {epoch+1}/{config['epochs']} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val MAE: {val_mae:.4f} | "
            f"Val RMSE: {val_rmse:.4f}"
        )

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse
        })

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= config["early_stopping_patience"]:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_state)

    return model, pd.DataFrame(history)

In [ ]:
def train_remoteclip_model(model, train_loader, val_loader, config):
    criterion = nn.L1Loss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )

    best_val_mae = float("inf")
    best_state = None
    patience_counter = 0
    history = []

    for epoch in range(config["epochs"]):
        model.train()
        train_loss = 0

        for batch in train_loader:
            optimizer.zero_grad()

            images = batch["image"].to(config["device"])
            texts = batch["text"]
            targets = batch["target"].to(config["device"])

            preds = model(images, texts)
            loss = criterion(preds, targets)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
            optimizer.step()

            train_loss += loss.item() * images.size(0)

        train_loss /= len(train_loader.dataset)

        val_loss, val_mae, val_rmse, _ = evaluate_remoteclip_model(model, val_loader, config)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse
        })

        print(
            f"Epoch {epoch+1}/{config['epochs']} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val MAE: {val_mae:.4f} | "
            f"Val RMSE: {val_rmse:.4f}"
        )

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse
        })

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= config["early_stopping_patience"]:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_state)

    return model, pd.DataFrame(history)

In [ ]:
wandb.login()

Image Only Model

In [ ]:
def run_swin_image_only():
    seed_everything(CONFIG["seed"])

    run_config = CONFIG.copy()
    run_config["experiment"] = "swinv2_image_only"
    run_config["fusion"] = "none"
    run_config["text_column"] = "none"
    run_config["text_model"] = "none"

    wandb.init(
        project="DI725-landcover-regression",
        name="swinv2_image_only",
        config=run_config,
        reinit=True
    )

    train_ds = LandCoverDataset(train_df, IMAGE_DIR, transform=train_tfms)
    val_ds = LandCoverDataset(val_df, IMAGE_DIR, transform=eval_tfms)
    test_ds = LandCoverDataset(test_df, IMAGE_DIR, transform=eval_tfms)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

    model = ImageOnlyModel(
        model_name=CONFIG["model_name"],
        num_outputs=len(TARGET_COLS)
    ).to(CONFIG["device"])

    model, history = train_image_model(model, train_loader, val_loader, CONFIG)

    test_loss, test_mae, test_rmse, class_mae = evaluate_image_model(model, test_loader, CONFIG)

    wandb.log({
        "test_loss": test_loss,
        "test_mae": test_mae,
        "test_rmse": test_rmse
    })

    wandb.log({
        "class_mae_table": wandb.Table(
            dataframe=pd.DataFrame({
                "class": TARGET_COLS,
                "class_mae": class_mae
            })
        )
    })

    wandb.finish()

    return {
        "experiment": "swinv2_image_only",
        "test_mae": test_mae,
        "test_rmse": test_rmse
    }

In [ ]:
def run_swin_transformer_text(text_col, text_model, text_scale=1.0):
    seed_everything(CONFIG["seed"])

    run_config = CONFIG.copy()
    run_config["experiment"] = "swinv2_transformer_text"
    run_config["text_column"] = text_col
    run_config["text_model"] = text_model
    run_config["text_scale"] = text_scale
    run_config["fusion"] = "gated_frozen_scaled"

    run_name = f"swinv2_{text_col}_{text_model.split('/')[-1]}_scale_{text_scale}"

    wandb.init(
        project="DI725-landcover-regression",
        name=run_name,
        config=run_config,
        reinit=True
    )

    tokenizer = AutoTokenizer.from_pretrained(text_model)

    train_ds = LandCoverDataset(train_df, IMAGE_DIR, transform=train_tfms, tokenizer=tokenizer, text_col=text_col)
    val_ds = LandCoverDataset(val_df, IMAGE_DIR, transform=eval_tfms, tokenizer=tokenizer, text_col=text_col)
    test_ds = LandCoverDataset(test_df, IMAGE_DIR, transform=eval_tfms, tokenizer=tokenizer, text_col=text_col)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

    model = ImageTextGatedFrozenScaledModel(
        image_model_name=CONFIG["model_name"],
        text_model_name=text_model,
        num_outputs=len(TARGET_COLS),
        text_scale=text_scale
    ).to(CONFIG["device"])

    model, history = train_text_model(model, train_loader, val_loader, CONFIG)

    test_loss, test_mae, test_rmse, class_mae = evaluate_text_model(model, test_loader, CONFIG)

    wandb.log({
        "test_loss": test_loss,
        "test_mae": test_mae,
        "test_rmse": test_rmse
    })

    wandb.log({
        "class_mae_table": wandb.Table(
            dataframe=pd.DataFrame({
                "class": TARGET_COLS,
                "class_mae": class_mae
            })
        )
    })

    wandb.finish()

    return {
        "experiment": run_name,
        "text_column": text_col,
        "text_model": text_model,
        "text_scale": text_scale,
        "test_mae": test_mae,
        "test_rmse": test_rmse
    }